In this notebook we bulk retrieve all URLs to annexed tables (with .xlsx suffix) in USGS Mineral Yearbook on its website.

Notably, USGS altered its website layout for Mineral Yearbooks, creating individual homepages for part of the countries while leaving others untouched, thus nullifying our previous workflow. The new version allows users to first check whether there are PDF and XLS/XLSX files in the regional homepage (which previouly aggregated all reports and tables). If they are missing for certain countries while the country name is attached with a hyperlink targeting its country-specific homepage, then we visit the homepage and recursively scrape URLs for PDF and XLS/XLSX files there. Nevertheless, it requires us to develop more adaptive and robust URL retrieval modules to fit different possible HTML layouts. 

*NOTE: Homepage layout for European countries is significantly different from other continents, which removes readily visible cutpoints embedded in \<h5\> labels.*

In [1]:
import requests
from bs4 import BeautifulSoup

## Request html from USGS website (Europe and Central Euroasia)
usgs_region_url = "https://www.usgs.gov/centers/national-minerals-information-center/europe-and-central-eurasia"
usgs_region_response = requests.get(url = usgs_region_url,
                                    headers= {
                                        'User-Agent': "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36 Edg/149.0.0.0"
                                    })

usgs_region_response.encoding = 'utf-8'
usgs_region_soup = BeautifulSoup(usgs_region_response.content, features='html.parser')

In [2]:
# Re-check
print(usgs_region_soup.prettify())

<!DOCTYPE html>
<html dir="ltr" lang="en" prefix="og: https://ogp.me/ns#">
 <head>
  <!-- Google Tag Manager -->
  <script>
   (function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({'gtm.start':
        new Date().getTime(),event:'gtm.js'});var f=d.getElementsByTagName(s)[0],
      j=d.createElement(s),dl=l!='dataLayer'?'&l='+l:'';j.async=true;j.src=
      'https://www.googletagmanager.com/gtm.js?id='+i+dl;f.parentNode.insertBefore(j,f);
    })(window,document,'script','dataLayer', 'GTM-KWRSKS');
  </script>
  <!-- End Google Tag Manager -->
  <meta charset="utf-8"/>
  <meta content="Reports on countries' mineral industries, including statistics on mineral production and trade, government policies, industry structure, and commodity sector developments." name="description"/>
  <meta content="Ukraine,Greenland, Netherlands" name="keywords"/>
  <link href="https://www.usgs.gov/centers/national-minerals-information-center/europe-and-central-eurasia" rel="canonical"/>
  <meta content="USGS" property

In [20]:
sections_list = usgs_region_soup.find_all('h2')
country_reports_section = [section for section in sections_list if 'country reports' in section.text.lower() and 'multi' not in section.text.lower()][0]
country_reports_section_list = country_reports_section.parent.find_all("li")

# Notice only a small proportions of countries in Europe and Central Euroasia do not have their own homepage. We tackle-and-remove with these exceptions first.
inpage_country_list = [list(field.children) for _, field in enumerate(country_reports_section_list) if 'Reports' in str(field)]

reports_dict = {}
tables_dict = {}
for block in inpage_country_list:
    country = block[0].get_text(strip=True)
    reports_dict[country] = {}
    tables_dict[country] = {}

    reports_list = block[1].find_all('a')
    for report in reports_list:
        year = report.get_text(strip=True)
        report_url = report.get('href', default='')
        reports_dict[country][year] = report_url
    tables_list = block[2].find_all('a')
    for table in tables_list:
        year = table.get_text(strip=True)
        # -----DEBUG: There are schema error in table labels--------
        if year == '200':
            year = '2005'
        if year == '5':
            continue
        # -----------------------------------------------------------
        table_url = table.get('href', default='')
        tables_dict[country][year] = table_url

In [ ]:
outpage_country_list

<li><a class="ck-anchor" id="al"></a><a class="ck-anchor" id="alb"></a><a href="https://usgs.gov/centers/national-minerals-information-center/albania">Albania</a></li>

In [30]:
# We then move on to other countries, all with individual homepages.
import sys; sys.path.append("../../modules/retrieve_url/")
import RetrieveCountrySite as rcs
import time, random
from urllib.parse import urljoin

outpage_country_list = [label for label in country_reports_section_list if label.find('p') is None]
outpage_country_list

[<li><a class="ck-anchor" id="al"></a><a class="ck-anchor" id="alb"></a><a href="https://usgs.gov/centers/national-minerals-information-center/albania">Albania</a></li>,
 <li><a class="ck-anchor" id="am"></a><a class="ck-anchor" id="arm"></a><a href="https://usgs.gov/centers/national-minerals-information-center/armenia">Armenia</a></li>,
 <li><a class="ck-anchor" id="au"></a><a class="ck-anchor" id="aut"></a><a href="https://usgs.gov/centers/national-minerals-information-center/austria">Austria</a></li>,
 <li><a class="ck-anchor" id="aj"></a><a class="ck-anchor" id="aze"></a><a href="https://usgs.gov/centers/national-minerals-information-center/azerbaijan">Azerbaijan</a></li>,
 <li><a class="ck-anchor" id="bo"></a><a class="ck-anchor" id="blr"></a><a href="https://usgs.gov/centers/national-minerals-information-center/belarus">Belarus</a> </li>,
 <li><a class="ck-anchor" id="be"></a><a class="ck-anchor" id="bel"></a><a href="https://usgs.gov/centers/national-minerals-information-center/

In [3]:
sections_list = usgs_region_soup.find_all('h2')
country_reports_section = [section for section in sections_list if 'country reports' in section.text.lower() and 'multi' not in section.text.lower()][0]
country_reports_section_list = list(country_reports_section.next_siblings)

# Notice only a small proportions of countries in Europe and Central Euroasia do not have their own homepage. We tackle-and-remove with these exceptions first.
inpage_country_list = [(country_reports_section_list[index-1], field, country_reports_section_list[index+1], index) for index, field in enumerate(country_reports_section_list) if 'Reports:' in str(field)]

reports_dict = {}
tables_dict = {}
for block in inpage_country_list:
    country = block[0].get_text(strip=True)
    reports_dict[country] = {}
    tables_dict[country] = {}

    reports_list = block[1].find_all('a')
    for report in reports_list:
        year = report.get_text(strip=True)
        report_url = report.get('href', default='')
        reports_dict[country][year] = report_url
    tables_list = block[2].find_all('a')
    for table in tables_list:
        year = table.get_text(strip=True)
        table_url = table.get('href', default='')
        tables_dict[country][year] = table_url

# We then move on to other countries, all with individual homepages.
import sys; sys.path.append("../../modules/retrieve_url/")
import RetrieveCountrySite as rcs
import time, random
from urllib.parse import urljoin
with requests.Session() as session:
    for slice in country_reports_section_list:
        a_labels_list = slice.find_all('a')
        if len(a_labels_list) == 3:
            if a_labels_list[2].get('href') is not None:
                country = a_labels_list[2].get_text(strip=True)
                country_url = a_labels_list[2].get('href')
                if country_url.startswith("//"): # DEBUG: URL to Latvia's homepage is incomplete: '//www.usgs.gov/centers/national-minerals-information-center/latvia'
                    country_url = urljoin("https://", country_url)
                print(f"Visiting homepage of [{country}]..."); start_time = time.perf_counter()
                try:
                    response = session.get(url=country_url,
                            headers={
                                'User-Agent': "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36 Edg/149.0.0.0"
                            },
                            timeout=(10,5)) # DEBUG: Connection timeout threshold is 10 seconds. Reading timeout is 5 seconds
                    reports_dict.update(rcs.reports_tables_url_session(country_response = response, country_url = country_url)[0]) # Invoke module for scraping country homepage
                    tables_dict.update(rcs.reports_tables_url_session(country_response = response, country_url = country_url)[1])
                    print(f"Scraping completed! Time consumed: {time.perf_counter()-start_time:.2f}s")
                except requests.exceptions.Timeout:
                    print("Failed to scrape! Error: Time out") # DEBUG: GET request could at times encounter delays of the server.
                sleep_time = 1 + random.uniform(0,2); time.sleep(sleep_time); print(f"Snap ended. Time consumed: {sleep_time:.2f}s")

IndexError: list index out of range

In [ ]:
tables_dict

In [ ]:
sections_list = usgs_region_soup.find_all('h2')
country_reports_section = [section for section in sections_list if 'country reports' in section.text.lower() and 'multi' not in section.text.lower()][0]
country_reports_section_iterator = country_reports_section.next_siblings

split_output = split_output[1:] # Remove chapeau

import pandas as pd
import sys; sys.path.append("../../modules/retrieve_url/")
import RetrieveCountrySite as rcs
import time
import random

reports_dict = {}
tables_dict = {}
for slice in split_output:
    country = slice[0].get_text(strip=True) # Retrieve country name from <h5> label at the beginning of every retained slice

    reports_index = [index for index, head in enumerate(slice) if 'Report' in list(head.children)[0]] # Retrieve "Report" block's index within each slice. A typical slice could contain a title block (<h5> label), a Report block (1st <p> label), a Table block (2nd <p> label) and line break block (3nd <p> label)
    tables_index = [index for index, head in enumerate(slice) if 'Table' in list(head.children)[0]]
    reports_index = reports_index[0] if len(reports_index) != 0 else None # Mark report_index as None if Report block does not apply to the country. Otherwise it would be an empty list (e.g., "[]").
    tables_index = tables_index[0] if len(tables_index) != 0 else None

    if reports_index is not None: # If Report block exist in the slice
        reports_dict[country] = {}
        reports_label_list = slice[reports_index].find_all('a') # Focus on the Report block and retain only items labeled <a> (container for URL and year info of the report)
        for report_label in reports_label_list:
            year = report_label.get_text(strip=True) # Retrieve year of the report (remove redundant linebreaks and blanks)
            report_url = report_label.get('href', default='') # Retrieve URL to the report
            if report_url.endswith('.pdf') and report_url.startswith(('http','https')): # Double check if the URL is suffixed with .pdf and preffixed with http(s), in order to avoid crash of programs in subsequent processes requiring batch read-in of the files.
                reports_dict[country][year] = report_url # Provisionally save to dictionary
    if tables_index is not None: # If Table block exist in the slice
        tables_dict[country] = {}
        tables_label_list = slice[tables_index].find_all('a')
        for table_lable in tables_label_list:
            year = table_lable.get_text(strip=True)
            table_url = table_lable.get('href', default='')
            if table_url.endswith(('.xls','.xlsx')) and table_url.startswith(('http','https')) and len(year) != 0:
                tables_dict[country][year] = table_url
    if reports_index is None and tables_index is None and slice[0].find('a') is not None: # IMPORTANT: Under USGS's new website layout, Reports and Tables of some countries are transferred to independent country homepages, causing absence of Report and Table block from continent homepage.
        country_url = None
        for label in slice[0].find_all('a'): # IMPORTANT DEBUG: We initially mistakenly wrote .find('a') instead of .find_all('a'), leaving it possible to return None.
            attempt = label.get('href')  # Try to get 'href' from label (None is returned if the field does not exist in the current label)
            if attempt is not None:
                country_url = attempt
        if country_url is None: # DEBUG: In North America's case, part of the <h5> labels contain no country-specific information (e.g., "<h5><a class="ck-anchor" id="bz"></a></h5>").
            print("Skip current <h5> slice. Reason: 'country_url' unspecified")
            continue
        print(f"Visiting homepage of [{country}]..."); start_time = time.perf_counter()
        try:
            reports_dict.update(rcs.reports_tables_url(country_name = country, country_url = country_url)[0]) # Invoke module for scraping country homepage
            tables_dict.update(rcs.reports_tables_url(country_name = country, country_url = country_url)[1])
            print(f"Scraping completed! Time consumed: {time.perf_counter()-start_time:.2f}s")
        except requests.exceptions.Timeout:
            print("Failed to scrape! Error: Time out") # DEBUG: GET request could at times encounter delays of the server.
        sleep_time = 3 + random.uniform(0,3); time.sleep(sleep_time); print(f"Snap ended. Time consumed: {sleep_time:.2f}s")

Skip current <h5> slice. Reason: 'country_url' unspecified
Skip current <h5> slice. Reason: 'country_url' unspecified
Skip current <h5> slice. Reason: 'country_url' unspecified
Skip current <h5> slice. Reason: 'country_url' unspecified
Skip current <h5> slice. Reason: 'country_url' unspecified
Skip current <h5> slice. Reason: 'country_url' unspecified
Skip current <h5> slice. Reason: 'country_url' unspecified
Skip current <h5> slice. Reason: 'country_url' unspecified


In [ ]:
# FOR DEBUGGING
split_output

[[<h5><a class="ck-anchor" id="bh"></a><strong>Belize</strong> </h5>,
  <p>Report: <a href="https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/1994/9525094.pdf">1994</a> | <a href="https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/1995/9538095.pdf">1995</a> | <a href="https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/1996/9538096.pdf">1996</a> | <a href="https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/1997/9538097.pdf">1997</a> | <a href="https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/1998/9538098.pdf">1998</a> | <a href="https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/1999/9538099.pdf">1999</a> </p>,
  <p>See Central America multi-country reports for recent years.</p>,
  <p> </p>],
 [<h5><a class="ck-anchor" id="bz"></a></h5>],
 [<

In [ ]:
# Re-check
reports_dict

{'Belize': {'1994': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/1994/9525094.pdf',
  '1995': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/1995/9538095.pdf',
  '1996': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/1996/9538096.pdf',
  '1997': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/1997/9538097.pdf',
  '1998': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/1998/9538098.pdf',
  '1999': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/1999/9538099.pdf'},
 'Canada': {'1994': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/1994/9505094.pdf',
  '1995': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/1995/9505095.pdf',
  '

In [ ]:
# Re-check
tables_dict

{'Canada': {'2002': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/2002/camyb02r2.xls',
  '2003': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/2003/camyb03.xls',
  '2004': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/2004/camyb04.xls',
  '2005': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/2005/camyb05.xls',
  '2006': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/2006/myb3-2006-ca.xls',
  '2007': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/2007/myb3-2007-ca.xls',
  '2008': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/2008/myb3-2008-ca.xls',
  '2009': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/2009/myb3-2009-c

In [24]:
# FOR DEBUGGING
slice = split_output[0]
print(slice)

[<h5><a class="ck-anchor" id="bh"></a><strong>Belize</strong> </h5>, <p>Report: <a href="https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/1994/9525094.pdf">1994</a> | <a href="https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/1995/9538095.pdf">1995</a> | <a href="https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/1996/9538096.pdf">1996</a> | <a href="https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/1997/9538097.pdf">1997</a> | <a href="https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/1998/9538098.pdf">1998</a> | <a href="https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/1999/9538099.pdf">1999</a> </p>, <p>See Central America multi-country reports for recent years.</p>, <p> </p>]


In [ ]:
# Write to CSV (Tables' URL)
import re
tables_df = pd.DataFrame(tables_dict).reset_index(names=['year'])
tables_df['year'] = tables_df['year'].str.replace(r'[a-zA-Z\s]', '', regex=True).str.strip('-').replace('2017-18','2018',regex=False).replace('2020-21','2021',regex=False)
tables_df = tables_df.melt(id_vars='year', var_name='country', value_name='table_url').sort_values(by=['country','year']).dropna()
tables_df.to_csv("../../data/original_url/tables_new/usgs_northamerica_tables.csv", index=False)

In [ ]:
# Write to CSV (Reports' URL)
import re
reports_df = pd.DataFrame(reports_dict).reset_index(names=['year'])
reports_df['year'] = reports_df['year'].str.replace(r'[a-zA-Z\s]', '', regex=True).str.strip('-').replace('2017-18','2018',regex=False).replace('2020-21','2021',regex=False)
reports_df = reports_df.melt(id_vars='year', var_name='country', value_name='report_url').sort_values(by=['country','year']).dropna()
reports_df.to_csv("../../data/original_url/reports_new/usgs_northamerica_reports.csv", index=False)